In [34]:
import cv2
import numpy as np
from ultralytics import YOLO

# ─── CONFIGURATION ──────────────────────────────────────────────────────────

VIDEO_PATH = "./speed_test_2.mp4"
OUTPUT_PATH = "./speed_output1.mp4"
MODEL_PATH = "./detection_track.pt"

CONF_THRES = 0.25
IOU_THRES = 0.45

cap = cv2.VideoCapture(VIDEO_PATH)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
cap.release()

# ─── SPEED CONFIG ───────────────────────────────────────────────────────────

LINE_Y1 = int(height * 0.70)
LINE_Y2 = int(height * 0.85)
METERS_BETWEEN_LINES = 8.0
LINE_COLOR_1 = (255, 100, 0)
LINE_COLOR_2 = (0, 100, 255)
TEXT_COLOR   = (0, 255, 80)
SPEED_PERSIST = 150

# ─── LANE CONFIG (curved road) ───────────────────────────────────────────────
# HOW TO CALIBRATE:
#   1. Run get_lane_points.py (below) to open your video frame
#   2. Click along each lane boundary line following the road curve
#   3. Copy the printed coordinates into LANE_LEFT_POLY and LANE_RIGHT_POLY
#
# Each polygon is a closed region — the area of ONE lane.
# Vehicles whose bottom-center falls OUTSIDE their lane polygon = violation.
#
# Example for a 2-lane curved road (replace with your actual coordinates):

LANE_POLYGONS = {
    "Lane 1": np.array([
        [int(width * 0.15), int(height * 0.26)],   # 🔽 lowered (was 0.22)
        [int(width * 0.30), int(height * 0.28)],
        [int(width * 0.39), int(height * 0.35)],   # ⬅️ more left (was 0.42)

        [int(width * 0.60), int(height * 0.70)],
        [int(width * 0.70), int(height * 0.95)],

        [int(width * 0.35), int(height * 0.95)],
        [int(width * 0.28), int(height * 0.70)],
    ], dtype=np.int32),

    "Lane 2": np.array([
        [int(width * 0.42), int(height * 0.35)],
        [int(width * 0.60), int(height * 0.28)],
        [int(width * 0.75), int(height * 0.25)],

        [int(width * 0.88), int(height * 0.65)],
        [int(width * 0.96), int(height * 0.95)],

        [int(width * 0.70), int(height * 0.95)],
        [int(width * 0.62), int(height * 0.70)],
    ], dtype=np.int32),
}

# How many frames a vehicle must be outside its lane to trigger a violation
# (avoids false positives on lane changes at borders)
VIOLATION_FRAME_THRESHOLD = 8

# ─── HELPERS ────────────────────────────────────────────────────────────────

def bottom_center(box):
    x1, y1, x2, y2 = box
    return int((x1 + x2) / 2), int(y2)

def crossed_line(prev_y, curr_y, line_y, tol=10):
    return (prev_y - line_y) * (curr_y - line_y) < 0 or abs(curr_y - line_y) < tol

def get_lane(point):
    """Returns the lane name the point belongs to, or None."""
    for lane_name, poly in LANE_POLYGONS.items():
        if cv2.pointPolygonTest(poly, point, False) >= 0:
            return lane_name
    return None

def draw_lane_overlays(frame):
    """Draw semi-transparent lane polygons on the frame."""
    overlay = frame.copy()
    colors = [(0, 255, 100), (0, 180, 255), (255, 200, 0)]
    for i, (lane_name, poly) in enumerate(LANE_POLYGONS.items()):
        color = colors[i % len(colors)]
        cv2.fillPoly(overlay, [poly], color)
        cv2.polylines(frame, [poly], isClosed=True, color=color, thickness=2)
        # Label the lane
        cx = int(np.mean(poly[:, 0]))
        cy = int(np.mean(poly[:, 1]))
        cv2.putText(frame, lane_name, (cx - 30, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    cv2.addWeighted(overlay, 0.15, frame, 0.85, 0, frame)

# ─── STATE ──────────────────────────────────────────────────────────────────

prev_positions     = {}
t1_timestamps      = {}
speed_labels       = {}
vehicle_lane       = {}       # track_id -> assigned lane (first seen lane)
outside_lane_count = {}       # track_id -> consecutive frames outside lane
violations         = {}       # track_id -> violation flag (bool)

# ─── CALIBRATION HELPER (run separately) ────────────────────────────────────
# Save this as get_lane_points.py and run it once to get your polygon coords:
CALIBRATION_CODE = '''
import cv2

VIDEO_PATH = "./speed_test_2.mp4"
points = []

def click(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        points.append([x, y])
        print(f"  [{x}, {y}],")
        cv2.circle(img, (x, y), 5, (0, 255, 0), -1)
        if len(points) > 1:
            cv2.line(img, tuple(points[-2]), tuple(points[-1]), (0,255,0), 2)
        cv2.imshow("frame", img)

cap = cv2.VideoCapture(VIDEO_PATH)
ret, img = cap.read()
cap.release()

cv2.imshow("frame", img)
cv2.setMouseCallback("frame", click)
print("Click lane boundary points. Press Q when done.")
while cv2.waitKey(0) != ord("q"):
    pass
cv2.destroyAllWindows()
print("Final points:", points)
'''

# ─── MAIN ───────────────────────────────────────────────────────────────────

def run():
    model = YOLO(MODEL_PATH)
    cap   = cv2.VideoCapture(VIDEO_PATH)

    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"Frame size: {width}x{height}, FPS: {fps}")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

    frame_idx = 0

    for result in model.track(
        source=VIDEO_PATH,
        conf=CONF_THRES,
        iou=IOU_THRES,
        tracker="bytetrack.yaml",
        persist=True,
        stream=True,
    ):
        frame     = result.orig_img.copy()
        timestamp = frame_idx / fps

        # ── Draw lane overlays ──────────────────────────────────
        draw_lane_overlays(frame)

        # ── Draw speed lines ────────────────────────────────────
        cv2.line(frame, (0, LINE_Y1), (width, LINE_Y1), LINE_COLOR_1, 2)
        cv2.line(frame, (0, LINE_Y2), (width, LINE_Y2), LINE_COLOR_2, 2)
        cv2.putText(frame, "Line 1", (10, LINE_Y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, LINE_COLOR_1, 2)
        cv2.putText(frame, "Line 2", (10, LINE_Y2 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, LINE_COLOR_2, 2)

        if result.boxes is not None and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids   = result.boxes.id.cpu().numpy().astype(int)

            for box, track_id in zip(boxes, ids):
                cx, cy = bottom_center(box)
                point  = (float(cx), float(cy))

                # ── Assign lane on first detection ──────────────
                current_lane = get_lane(point)

                if track_id not in vehicle_lane:
                    if current_lane is not None:
                        vehicle_lane[track_id]       = current_lane
                        outside_lane_count[track_id] = 0
                        violations[track_id]         = False

                # ── Lane violation check ─────────────────────────
                if track_id in vehicle_lane:
                    assigned_lane = vehicle_lane[track_id]

                    if current_lane != assigned_lane:
                        outside_lane_count[track_id] = \
                            outside_lane_count.get(track_id, 0) + 1
                    else:
                        outside_lane_count[track_id] = 0  # reset if back in lane

                    if outside_lane_count.get(track_id, 0) >= VIOLATION_FRAME_THRESHOLD:
                        violations[track_id] = True
                        print(f"[VIOLATION] ID {track_id} left {assigned_lane}")

                # ── Speed estimation ─────────────────────────────
                if track_id in prev_positions:
                    _, prev_cy = prev_positions[track_id]

                    if cy > prev_cy:
                        if crossed_line(prev_cy, cy, LINE_Y1):
                            t1_timestamps[track_id] = timestamp

                        if crossed_line(prev_cy, cy, LINE_Y2):
                            if track_id in t1_timestamps:
                                dt = abs(timestamp - t1_timestamps.pop(track_id))
                                if dt > 0:
                                    speed = (METERS_BETWEEN_LINES / dt) * 3.6
                                    speed_labels[track_id] = {
                                        "speed_kmh": speed,
                                        "ttl": SPEED_PERSIST,
                                    }
                                    print(f"[SPEED] ID {track_id} → {speed:.2f} km/h")

                prev_positions[track_id] = (cx, cy)

                # ── Draw bounding box ─────────────────────────────
                x1, y1, x2, y2 = map(int, box)
                is_violation    = violations.get(track_id, False)
                box_color       = (0, 0, 255) if is_violation else (0, 200, 255)
                label_text      = f"ID:{track_id}"
                if is_violation:
                    label_text += " [LANE VIOL]"

                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                cv2.putText(frame, label_text, (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 1)

                # ── Draw lane assignment label ────────────────────
                if track_id in vehicle_lane:
                    cv2.putText(frame, vehicle_lane[track_id], (x1, y2 + 15),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)

                cv2.circle(frame, (cx, cy), 4, (0, 255, 255), -1)

        # ── Draw speed labels ────────────────────────────────────
        expired = []
        for tid, info in speed_labels.items():
            if tid in prev_positions:
                cx, cy    = prev_positions[tid]
                speed     = info["speed_kmh"]
                label     = f"{speed:.1f} km/h"
                y_text    = max(30, cy - 20)
                color     = (0, 0, 255) if speed > 50 else TEXT_COLOR
                thickness = 4 if speed > 50 else 2
                cv2.putText(frame, label, (cx, y_text),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), thickness + 2)
                cv2.putText(frame, label, (cx, y_text),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, thickness)
            info["ttl"] -= 1
            if info["ttl"] <= 0:
                expired.append(tid)
        for tid in expired:
            del speed_labels[tid]

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f"\n✅ Done! Output saved to: {OUTPUT_PATH}")



In [35]:
def preview_frame():
    cap = cv2.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    cap.release()

    if not ret:
        print("❌ Failed to read video")
        return

    # Draw lanes
    draw_lane_overlays(frame)

    # Draw speed lines
    cv2.line(frame, (0, LINE_Y1), (width, LINE_Y1), LINE_COLOR_1, 2)
    cv2.line(frame, (0, LINE_Y2), (width, LINE_Y2), LINE_COLOR_2, 2)

    cv2.putText(frame, f"LINE_Y1: {LINE_Y1}", (10, LINE_Y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, LINE_COLOR_1, 2)
    cv2.putText(frame, f"LINE_Y2: {LINE_Y2}", (10, LINE_Y2 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, LINE_COLOR_2, 2)

    cv2.imshow("Lane Calibration Preview", frame)
    print("👉 Press any key to close preview...")
    cv2.waitKey(0)
    cv2.destroyAllWindows()
preview_frame()

👉 Press any key to close preview...


In [36]:

if __name__ == "__main__":
    run()

Frame size: 1920x1080, FPS: 50.0

video 1/1 (frame 1/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\lane_change\speed_test_2.mp4: 384x640 7 cars, 20.9ms
video 1/1 (frame 2/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\lane_change\speed_test_2.mp4: 384x640 7 cars, 17.4ms
video 1/1 (frame 3/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\lane_change\speed_test_2.mp4: 384x640 6 cars, 1 scooter, 17.3ms
video 1/1 (frame 4/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\lane_change\speed_test_2.mp4: 384x640 6 cars, 17.3ms
video 1/1 (frame 5/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\lane_change\speed_test_2.mp4: 384x640 6 cars, 1 scooter, 17.1ms
video 1/1 (frame 6/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\lane_change\speed_test_2.mp4: 384x640 6 cars, 1 scooter, 17.1ms
video 1/1 (frame 7/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\lane_change\speed_test_2.mp4: 384x640 6 cars, 16.9ms
video 1/1 (frame 8/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\lane_change\speed_